# models

> Verify step state and verification result models for Phase 4: Verify

In [ ]:
#| default_exp models

In [ ]:
#| export
from typing import Optional, List
from typing_extensions import TypedDict
from dataclasses import dataclass, field

## VerifyStepState

TypedDict for Phase 4: Verify step state.

In [ ]:
#| export
class VerifyStepState(TypedDict, total=False):
    """State for Phase 4: Verify."""
    
    document_id: str  # UUID of committed Document node (required)
    media_path: Optional[str]  # Source media path for display context

## SegmentSample

Lightweight segment data for sample display.

In [ ]:
#| export
@dataclass
class SegmentSample:
    """Lightweight segment data for sample display."""
    
    index: int  # Segment index in document
    text: str  # Truncated text (~60 chars)
    start_time: Optional[float] = None  # Start time in seconds
    end_time: Optional[float] = None  # End time in seconds
    
    @property
    def duration(self) -> Optional[float]:  # Computed duration in seconds
        """Compute duration from start and end times."""
        if self.start_time is not None and self.end_time is not None:
            return self.end_time - self.start_time
        return None

## VerificationResult

Complete verification data computed from graph database queries.

In [ ]:
#| export
@dataclass
class VerificationResult:
    """Complete verification data from graph database queries."""
    
    # Document info (from graph)
    document_id: str  # UUID of Document node
    document_title: str  # Document title
    document_media_type: str  # Media type ('audio', 'video', 'text')
    
    # Segment stats (computed)
    segment_count: int  # Total number of segments
    total_duration: float  # Total duration in seconds
    avg_segment_duration: float  # Average segment duration in seconds
    
    # Integrity checks - STARTS_WITH
    has_starts_with: bool  # Whether STARTS_WITH edge exists
    starts_with_count: int  # Number of STARTS_WITH edges (should be 1)
    
    # Integrity checks - NEXT chain
    next_chain_complete: bool  # Whether NEXT chain is complete
    next_count: int  # Number of NEXT edges (should be segment_count - 1)
    
    # Integrity checks - PART_OF
    part_of_complete: bool  # Whether all PART_OF edges exist
    part_of_count: int  # Number of PART_OF edges (should be segment_count)
    
    # Integrity checks - timing
    all_have_timing: bool  # Whether all segments have timing
    segments_missing_timing: int  # Count of segments without timing
    
    # Integrity checks - sources
    all_have_sources: bool  # Whether all segments have sources
    segments_missing_sources: int  # Count of segments without sources
    
    # Source info
    source_plugins: List[str] = field(default_factory=list)  # Unique plugin names
    
    # Sample segments
    first_segments: List[SegmentSample] = field(default_factory=list)  # First 3 segments
    last_segments: List[SegmentSample] = field(default_factory=list)  # Last 3 segments
    
    @property
    def all_checks_passed(self) -> bool:  # Whether all integrity checks passed
        """Check if all integrity checks passed."""
        return (
            self.has_starts_with and
            self.starts_with_count == 1 and
            self.next_chain_complete and
            self.part_of_complete and
            self.all_have_timing and
            self.all_have_sources
        )

## VerifyUrls

URL bundle for Phase 4 verify route handlers.

In [ ]:
#| export
@dataclass
class VerifyUrls:
    """URL bundle for Phase 4 verify route handlers."""
    
    # Verification routes
    verify: str = ""  # Main verification computation route
    sample: str = ""  # Jump-to-index sample fetch route

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()